# HyperProvider Output Field Inspection

这个 notebook 用来查看 `HyperProvider.fetch_daily()` 当前会输出哪些字段，并区分：

- 核心字段：固定来自同一个数据源
- 补充字段：由其他数据源按需补入

In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import free_market_data.providers.hyper_provider as hyper_provider_module

importlib.reload(hyper_provider_module)
HyperProvider = hyper_provider_module.HyperProvider

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

In [3]:
from free_market_data.providers import DEFAULT_PROVIDER_CLASSES

provider_map = {name: cls() for name, cls in DEFAULT_PROVIDER_CLASSES.items()}

hyper_provider = HyperProvider(
    provider_map=provider_map,
    provider_order=('akshare', 'baostock', 'yahoo', 'tencent', 'xueqiu', 'sina', 'sohu'),
)

stock_code = '600519.SH'
start_date = pd.Timestamp('2024-01-01')
end_date = pd.Timestamp('2024-12-31')

In [6]:
daily = hyper_provider.fetch_daily(
    code=stock_code,
    start_date=start_date,
    end_date=end_date
)

print(f'rows: {len(daily)}')
daily.head()

rows: 242


,date,stock_code,open,close,high,low,volume,source,qfq_factor,hfq_factor,price_source,updated_at,adj_close,change,amount
0,2024-01-02,600519.SH,1715.00,1685.01,1718.19,1678.10,32156.0,"tencent,yahoo,sohu",0.936905,6.249298,tencent,2026-05-11 15:56:37,1567.384155,-40.99,544008.25
1,2024-01-03,600519.SH,1681.11,1694.00,1695.22,1676.33,20229.0,"tencent,yahoo,sohu",0.937240,6.245999,tencent,2026-05-11 15:56:37,1575.746704,8.99,341140.06
2,2024-01-04,600519.SH,1693.00,1669.00,1693.00,1662.93,21551.0,"tencent,yahoo,sohu",0.936300,6.255260,tencent,2026-05-11 15:56:37,1552.491699,-25.0,360397.0
3,2024-01-05,600519.SH,1661.33,1663.36,1678.66,1652.11,20243.0,"tencent,yahoo,sohu",0.936084,6.257387,tencent,2026-05-11 15:56:37,1547.245483,-5.64,337315.56
4,2024-01-08,600519.SH,1661.00,1643.99,1662.00,1640.01,25586.0,"tencent,yahoo,sohu",0.935331,6.264806,tencent,2026-05-11 15:56:37,1529.227539,-19.37,421191.84


In [7]:
core_fields = [
    'date',
    'stock_code',
    'open',
    'high',
    'low',
    'close',
    'qfq_factor',
    'hfq_factor',
    'price_source',
    'source',
    'updated_at',
]

all_fields = list(daily.columns)
extra_fields = [column for column in all_fields if column not in core_fields]

summary = pd.DataFrame({
    'field': all_fields,
    'is_core': [column in core_fields for column in all_fields],
    'non_null_count': [int(daily[column].notna().sum()) for column in all_fields],
    'dtype': [str(daily[column].dtype) for column in all_fields],
})

print('core fields:')
print(core_fields)
print('\nextra fields:')
print(extra_fields)
summary

core fields:
['date', 'stock_code', 'open', 'high', 'low', 'close', 'qfq_factor', 'hfq_factor', 'price_source', 'source', 'updated_at']

extra fields:
['volume', 'adj_close', 'change', 'amount']


,field,is_core,non_null_count,dtype
0,date,True,242,datetime64[us]
1,stock_code,True,242,str
2,open,True,242,float64
3,close,True,242,float64
4,high,True,242,float64
5,low,True,242,float64
6,volume,False,242,float64
7,source,True,242,object
8,qfq_factor,True,242,float64
9,hfq_factor,True,242,float64


In [ ]:
if 'price_source' in daily.columns:
    print('price_source values:')
    print(daily['price_source'].dropna().unique().tolist())

if 'source' in daily.columns:
    print('source values:')
    print(daily['source'].dropna().unique().tolist())

daily.tail()

## Realtime Demo

这一节演示 `HyperProvider.fetch_realtime()` 的即时行情返回结果。

目标是直接查看：
- 当前返回了哪些实时字段
- 是否包含价格、成交量、时间、来源
- 返回结果长什么样

In [12]:
realtime_codes = ['600519.SH', '000001.SZ']

realtime = hyper_provider.fetch_realtime(realtime_codes)

realtime_fields = list(realtime.columns)
print('realtime fields:')
print(realtime_fields)
print(f'rows: {len(realtime)}')

realtime

Exception ignored in: <function tqdm.__del__ at 0x00000212E0B2B2E0>
Traceback (most recent call last):
  File "d:\2026_claude\StockFrame\.venv\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "d:\2026_claude\StockFrame\.venv\Lib\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


realtime fields:
['stock_code', 'price', 'open', 'high', 'low', 'pre_close', 'volume', 'timestamp', 'source']
rows: 2


,stock_code,price,open,high,low,pre_close,volume,timestamp,source
0,600519.SH,NaN,1372.890015,None,None,None,NaN,2026-05-11 16:08:30.151466,yahoo
1,000001.SZ,NaN,11.340000,None,None,None,NaN,2026-05-11 16:08:30.431359,yahoo
